In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import numpy as np

In [2]:
GAMMA = 0.99
LAMBDA = 0.95
EPSILON = 0.2

LR = 3e-3
ROLLOUT_STEPS = 500
UPDATES = 150

In [3]:
class ActorCritic(nn.Module):

    def __init__(self, state_dim, action_dim):
        super().__init__()

        self.body = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.Tanh()
        )

        self.actor = nn.Linear(64, action_dim)
        self.critic = nn.Linear(64, 1)

    def forward(self, x):

        x = self.body(x)

        action_logits = self.actor(x)
        value = self.critic(x).squeeze(-1)

        return action_logits, value

In [4]:
def compute_gae(rewards, values, next_value, dones):

    values = values + [next_value]

    advantages = [0] * len(rewards)

    gae = 0

    for t in reversed(range(len(rewards))):

        mask = 1 - dones[t]

        delta = rewards[t] + GAMMA * values[t + 1] * mask - values[t]

        gae = delta + GAMMA * LAMBDA * mask * gae

        advantages[t] = gae

    returns = []

    for i in range(len(rewards)):
        returns.append(advantages[i] + values[i])

    return advantages, returns

In [5]:
def train():

    env = gym.make("CartPole-v1")

    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    model = ActorCritic(state_dim, action_dim)

    optimizer = optim.Adam(model.parameters(), lr=LR)

    state, _ = env.reset()

    for update in range(UPDATES):

        states = []
        actions = []
        rewards = []
        dones = []
        old_log_probs = []
        values = []

        # Collect experience
        for step in range(ROLLOUT_STEPS):

            state_tensor = torch.FloatTensor(state)

            logits, value = model(state_tensor)

            dist = Categorical(logits=logits)

            action = dist.sample()

            next_state, reward, terminated, truncated, _ = env.step(action.item())

            done = terminated or truncated

            states.append(state)
            actions.append(action.item())
            rewards.append(reward)
            dones.append(done)

            old_log_probs.append(
                dist.log_prob(action).item()
            )

            values.append(value.item())

            if done:
                state, _ = env.reset()
            else:
                state = next_state

        # Get value of next state
        with torch.no_grad():

            state_tensor = torch.FloatTensor(state)

            _, next_value = model(state_tensor)

        # Calculate GAE
        advantages, returns = compute_gae(
            rewards,
            values,
            next_value.item(),
            dones
        )

        # Convert to tensors
        states_tensor = torch.FloatTensor(
            np.array(states)
        )

        actions_tensor = torch.LongTensor(actions)

        old_log_probs_tensor = torch.FloatTensor(
            old_log_probs
        )

        returns_tensor = torch.FloatTensor(
            returns
        )

        advantages_tensor = torch.FloatTensor(
            advantages
        )

        # Normalize advantages
        advantages_tensor = (
            advantages_tensor - advantages_tensor.mean()
        ) / (
            advantages_tensor.std() + 1e-8
        )

        # PPO update
        logits, values_pred = model(states_tensor)

        dist = Categorical(logits=logits)

        new_log_probs = dist.log_prob(actions_tensor)

        entropy = dist.entropy().mean()

        # Probability ratio
        ratio = torch.exp(
            new_log_probs - old_log_probs_tensor
        )

        # Surrogate losses
        surr1 = ratio * advantages_tensor

        surr2 = torch.clamp(
            ratio,
            1 - EPSILON,
            1 + EPSILON
        ) * advantages_tensor

        policy_loss = -torch.min(
            surr1,
            surr2
        ).mean()

        # Clipped value loss
        value_clipped = returns_tensor + torch.clamp(
            values_pred - returns_tensor,
            -EPSILON,
            EPSILON
        )

        value_loss = 0.5 * torch.max(
            (values_pred - returns_tensor) ** 2,
            (value_clipped - returns_tensor) ** 2
        ).mean()

        # Total loss
        loss = (
            policy_loss
            + 0.5 * value_loss
            - 0.01 * entropy
        )

        # Backpropagation
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        # Print results
        if update % 10 == 0:

            episode_count = max(1, sum(dones))

            average_reward = sum(rewards) / episode_count

            print(
                "Update",
                update,
                "| avg reward:",
                round(average_reward, 1),
                "| entropy:",
                round(entropy.item(), 3)
            )

    env.close()

    return model

In [6]:
model = train()

Update 0 | avg reward: 29.4 | entropy: 0.683
Update 10 | avg reward: 45.5 | entropy: 0.646
Update 20 | avg reward: 55.6 | entropy: 0.628
Update 30 | avg reward: 55.6 | entropy: 0.624
Update 40 | avg reward: 100.0 | entropy: 0.629
Update 50 | avg reward: 125.0 | entropy: 0.629
Update 60 | avg reward: 125.0 | entropy: 0.614
Update 70 | avg reward: 125.0 | entropy: 0.608
Update 80 | avg reward: 100.0 | entropy: 0.608
Update 90 | avg reward: 125.0 | entropy: 0.603
Update 100 | avg reward: 100.0 | entropy: 0.607
Update 110 | avg reward: 125.0 | entropy: 0.618
Update 120 | avg reward: 125.0 | entropy: 0.612
Update 130 | avg reward: 125.0 | entropy: 0.646
Update 140 | avg reward: 62.5 | entropy: 0.631


In [7]:
env = gym.make("CartPole-v1", render_mode="human")

state, _ = env.reset()

total_reward = 0

for step in range(500):

    state_tensor = torch.FloatTensor(state)

    with torch.no_grad():

        logits, _ = model(state_tensor)

    action = torch.argmax(logits).item()

    state, reward, terminated, truncated, _ = env.step(action)

    total_reward += reward

    if terminated or truncated:
        break

print("Test Reward:", total_reward)

env.close()

Test Reward: 188.0
